# Heart Disease Prediction — Hyperparameter Tuning

## Objective

Improve and compare Logistic Regression and Random Forest using cross-validation and a predefined hyperparameter search.

## Method

* Use a stratified 80/20 train/test split with a fixed random seed.
* Tune hyperparameters using only the training set.
* Compare candidates using ROC-AUC, while also reviewing precision, recall and F1-score.
* Evaluate the selected candidate on the held-out test set only after model selection.
* Preserve the full preprocessing and classification pipeline for reproducible inference.

The held-out test set is not used during hyperparameter selection.

In [2]:
#Loading the data and define preprocessing again

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol",
    "fbs", "restecg", "thalach", "exang",
    "oldpeak", "slope", "ca", "thal", "num",
]

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data",
    header=None,
    names=COLUMNS,
    na_values=["?"],
    skipinitialspace=True,
)

for col in COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["num"]).copy()
df["target"] = (df["num"] > 0).astype(int)

X = df.drop(columns=["num", "target"])
y = df["target"]

numeric_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]
categorical_features = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "ca", "thal"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print("Project root:", PROJECT_ROOT)
print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Data and preprocessing ready.")

Project root: c:\Users\Himaanish\Documents\heart-disease-mlops
Training records: 242
Test records: 61
Data and preprocessing ready.


In [3]:
#Setting up the hyperparameter searches

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42,
    )),
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        class_weight=None,
    )),
])

searches = {
    "Logistic Regression": GridSearchCV(
        estimator=lr_pipeline,
        param_grid={
            "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
            "classifier__class_weight": [None, "balanced"],
        },
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
        },
        refit="roc_auc",
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    ),

    "Random Forest": GridSearchCV(
        estimator=rf_pipeline,
        param_grid={
            "classifier__n_estimators": [100, 300],
            "classifier__max_depth": [None, 5, 10],
            "classifier__min_samples_leaf": [1, 2, 4],
            "classifier__class_weight": [None, "balanced"],
        },
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
        },
        refit="roc_auc",
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    ),
}

print("Hyperparameter searches configured.")

Hyperparameter searches configured.


In [4]:
#Running the searches

tuning_results = []

for name, search in searches.items():
    print(f"\nTuning {name}...")
    search.fit(X_train, y_train)

    best_index = search.best_index_
    tuning_results.append({
        "model": name,
        "best_cv_roc_auc": search.best_score_,
        "cv_roc_auc_std": search.cv_results_[
            "std_test_roc_auc"
        ][best_index],
        "cv_accuracy": search.cv_results_[
            "mean_test_accuracy"
        ][best_index],
        "cv_precision": search.cv_results_[
            "mean_test_precision"
        ][best_index],
        "cv_recall": search.cv_results_[
            "mean_test_recall"
        ][best_index],
        "cv_f1": search.cv_results_[
            "mean_test_f1"
        ][best_index],
        "best_parameters": search.best_params_,
    })

    print("Best CV ROC-AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

tuning_results_df = pd.DataFrame(tuning_results)
display(tuning_results_df.round(3))


Tuning Logistic Regression...
Best CV ROC-AUC: 0.9028
Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}

Tuning Random Forest...
Best CV ROC-AUC: 0.9034
Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 4, 'classifier__n_estimators': 300}


,model,best_cv_roc_auc,cv_roc_auc_std,cv_accuracy,cv_precision,cv_recall,cv_f1,best_parameters
0,Logistic Regression,0.903,0.017,0.830,0.823,0.810,0.813,"{'classifier__C': 0.1, 'classifier__class_weig..."
1,Random Forest,0.903,0.029,0.801,0.788,0.783,0.782,"{'classifier__class_weight': 'balanced', 'clas..."


In [5]:
#Evaluating the tuned models on the test set

test_results = []
best_models = {}

for name, search in searches.items():
    model = search.best_estimator_
    best_models[name] = model

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    test_results.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "test_recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "test_f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "test_roc_auc": roc_auc_score(
            y_test, probabilities
        ),
    })

    print(f"\n{name}")
    print(classification_report(
        y_test,
        predictions,
        target_names=["No disease", "Disease present"],
        zero_division=0,
    ))

tuned_test_results_df = pd.DataFrame(test_results)
display(tuned_test_results_df.round(3))


Logistic Regression
                 precision    recall  f1-score   support

     No disease       0.93      0.85      0.89        33
Disease present       0.84      0.93      0.88        28

       accuracy                           0.89        61
      macro avg       0.89      0.89      0.89        61
   weighted avg       0.89      0.89      0.89        61


Random Forest
                 precision    recall  f1-score   support

     No disease       0.93      0.82      0.87        33
Disease present       0.81      0.93      0.87        28

       accuracy                           0.87        61
      macro avg       0.87      0.87      0.87        61
   weighted avg       0.88      0.87      0.87        61



,model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,Logistic Regression,0.885,0.839,0.929,0.881,0.965
1,Random Forest,0.869,0.812,0.929,0.867,0.957
